# Structural Probe — UUAS on Penn Treebank

Implements the **Structural Distance Probe** from:
> *Mechanisms vs. Outcomes: Probing for Syntax Fails to Explain Performance on Targeted Syntactic Evaluations* (Agarwal et al., EMNLP 2025)

Originally proposed by Hewitt & Manning (2019).

**What it does:**
- Learns a linear map `B ∈ R^{k×n}` such that `‖B(hᵢ − hⱼ)‖²` approximates the tree distance between words i and j.
- Builds a Minimum Spanning Tree from predicted distances, then computes UUAS vs gold edges.

**Data:** UD English EWT (freely available). Swap in PTB if you have a license.


## 1. Setup

In [ ]:
import os, math, urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from transformers import AutoTokenizer, AutoModel
from conllu import parse_incr
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Download UD English EWT

The paper trains on Penn Treebank (PTB) parsed by Stanza into UD format.  
We use **UD English EWT** as a freely available substitute — the probe pipeline is identical.

To use PTB instead, set `DATA_DIR` to your PTB CoNLL-U files.

In [ ]:
DATA_DIR = Path("data/ud_ewt")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master"
SPLITS = {
    "train": "en_ewt-ud-train.conllu",
    "dev":   "en_ewt-ud-dev.conllu",
    "test":  "en_ewt-ud-test.conllu",
}

for split, fname in SPLITS.items():
    dest = DATA_DIR / fname
    if not dest.exists():
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(f"{BASE_URL}/{fname}", dest)
    else:
        print(f"{fname} already present.")

## 3. Parse CoNLL-U → (words, head indices, tree distances)

For each sentence we need:
- `words`: list of word strings (no multi-word tokens)
- `heads`: integer head index for each word (0 = ROOT)
- `dist_matrix`: `N×N` matrix where `d[i,j]` = tree distance between words i and j

In [ ]:
def tree_distances(heads):
    """Compute all-pairs tree distances from head array (1-indexed, 0=ROOT).
    
    Uses BFS from every node over the undirected dependency tree.
    Returns an N×N numpy array.
    """
    N = len(heads)
    # Build adjacency list (undirected, 0-indexed)
    adj = [[] for _ in range(N)]
    for i, h in enumerate(heads):
        j = h - 1  # convert to 0-indexed; -1 means ROOT (ignore)
        if j >= 0:
            adj[i].append(j)
            adj[j].append(i)

    dist = np.full((N, N), fill_value=np.inf)
    np.fill_diagonal(dist, 0)

    from collections import deque
    for src in range(N):
        visited = {src}
        queue = deque([(src, 0)])
        while queue:
            node, d = queue.popleft()
            dist[src, node] = d
            for nb in adj[node]:
                if nb not in visited:
                    visited.add(nb)
                    queue.append((nb, d + 1))
    return dist


def gold_edges(heads):
    """Return set of undirected gold edges {frozenset(i,j)}, 0-indexed, excluding ROOT."""
    edges = set()
    for i, h in enumerate(heads):
        j = h - 1
        if j >= 0:  # skip root relation
            edges.add(frozenset([i, j]))
    return edges


def load_conllu(path, max_len=50):
    """Load a CoNLL-U file. Returns list of (words, heads, dist_matrix, gold_edges).
    
    Filters out sentences longer than max_len and skips multi-word tokens.
    Punctuation is kept (paper excludes from UUAS evaluation, not from training).
    """
    sentences = []
    with open(path, encoding="utf-8") as f:
        for sent in parse_incr(f):
            # Skip multi-word tokens (id is a tuple like (1,2))
            tokens = [t for t in sent if isinstance(t["id"], int)]
            if not tokens or len(tokens) > max_len:
                continue
            words = [t["form"] for t in tokens]
            heads = [t["head"] for t in tokens]  # 0 = ROOT
            upos  = [t["upos"] for t in tokens]
            dists = tree_distances(heads)
            edges = gold_edges(heads)
            sentences.append((words, heads, dists, edges, upos))
    return sentences


print("Loading data...")
train_data = load_conllu(DATA_DIR / SPLITS["train"])
dev_data   = load_conllu(DATA_DIR / SPLITS["dev"])
test_data  = load_conllu(DATA_DIR / SPLITS["test"])

print(f"Train: {len(train_data)} | Dev: {len(dev_data)} | Test: {len(test_data)} sentences")

## 4. Load Model and Extract Hidden States

Key preprocessing from the paper (Section 3):
> "To reduce the influence of a word's semantic properties, we subtract word embeddings from each layer's contextual states:"
> $h^\ell_i = h^\ell_i - h^0_i \quad \forall \ell \in \{1, \ldots, L\}$

Subword → word alignment: we **average all subword tokens** per word, matching the
original Hewitt & Manning implementation (`np.mean(features[start:end+1], axis=0)`).

In [ ]:
# --- Config ---
MODEL_NAME = "gpt2"   # swap in any HuggingFace model
BATCH_SIZE_EXTRACT = 8

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
model = model.to(DEVICE).eval()

# GPT-2 has no pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Layers: {N_LAYERS}, Hidden dim: {HIDDEN_DIM}")

In [ ]:
def align_subwords_to_words(tokenizer, words, hidden_states):
    """Map subword hidden states back to word-level by AVERAGING all subwords per word.

    Matches the original Hewitt & Manning (2019) implementation:
        single_layer_features = np.mean(features[start:end+1], axis=0)

    Returns:
        list of tensors [n_words, hidden_dim] for each layer (layer 0 = embedding),
        or None if alignment fails.
    """
    enc = tokenizer(
        words, is_split_into_words=True,
        return_tensors="pt", add_special_tokens=True,
        truncation=True, max_length=512,
    )
    word_ids = enc.word_ids()  # token index → word index (None for special tokens)

    # Collect token indices for each word
    word_to_tokens = {}  # word_idx -> list of token indices
    for tok_idx, wid in enumerate(word_ids):
        if wid is not None:
            word_to_tokens.setdefault(wid, []).append(tok_idx)

    if len(word_to_tokens) != len(words):
        return None  # alignment failed

    # For each layer, average the subword token hidden states per word
    word_states = []
    for layer_hs in hidden_states:      # layer_hs: (1, seq_len, hidden_dim)
        hs = layer_hs[0]                # (seq_len, hidden_dim)
        averaged = torch.stack([
            hs[word_to_tokens[i]].mean(0)   # mean over subword tokens for word i
            for i in range(len(words))
        ])                              # (n_words, hidden_dim)
        word_states.append(averaged)

    return word_states  # list of L+1 tensors, each (n_words, hidden_dim)


@torch.no_grad()
def extract_hidden_states(data, tokenizer, model, device, desc="Extracting"):
    """Extract word-level hidden states for all sentences.

    Returns list of (words, heads, dists, edges, upos, word_states) where
    word_states is a list of L+1 tensors [0=embed, 1..L=layers], each (n_words, hidden_dim),
    with the embedding layer subtracted from all contextual layers.
    """
    results = []
    for words, heads, dists, edges, upos in tqdm(data, desc=desc):
        enc = tokenizer(
            words, is_split_into_words=True,
            return_tensors="pt", add_special_tokens=True,
            truncation=True, max_length=512,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        hidden_states = out.hidden_states  # tuple: (embed, L1, ..., LL)

        word_states = align_subwords_to_words(tokenizer, words, hidden_states)
        if word_states is None:
            continue

        # Subtract embedding layer (layer 0) from all contextual layers
        embed = word_states[0].cpu()
        subtracted = [embed] + [ws.cpu() - embed for ws in word_states[1:]]
        results.append((words, heads, dists, edges, upos, subtracted))

    return results


# Extract hidden states — cache to disk to avoid re-running on every notebook restart
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
cache_file = CACHE_DIR / f"{MODEL_NAME.replace('/', '_')}_hidden_states.pt"

if cache_file.exists():
    print("Loading cached hidden states...")
    train_hs, dev_hs, test_hs = torch.load(cache_file, weights_only=False)
else:
    train_hs = extract_hidden_states(train_data, tokenizer, model, DEVICE, "Train")
    dev_hs   = extract_hidden_states(dev_data,   tokenizer, model, DEVICE, "Dev")
    test_hs  = extract_hidden_states(test_data,  tokenizer, model, DEVICE, "Test")
    torch.save((train_hs, dev_hs, test_hs), cache_file)
    print(f"Saved to {cache_file}")

print(f"Train: {len(train_hs)} | Dev: {len(dev_hs)} | Test: {len(test_hs)} sentences")

## 5. Structural Probe

$$g^\text{struct}_\phi(h) = B^\text{struct} h, \quad B^\text{struct} \in \mathbb{R}^{k \times n}$$

**Loss (Eq. 1 in paper):**
$$\min_\phi \sum_S \frac{1}{|S|^2} \sum_{i,j} \left| d_{ij} - \|g_\phi(h_i - h_j)\|_2^2 \right|$$

**UUAS:** build MST from predicted distance matrix, compare edge sets with gold.

In [ ]:
class StructuralProbe(nn.Module):
    """Learns B ∈ R^{n×k} so that ‖B^T(hᵢ−hⱼ)‖² ≈ tree_distance(i, j).

    Matches the original Hewitt & Manning (2019) implementation exactly:
      - Parameter shape: (hidden_dim, probe_rank)  [n × k]
      - Forward:  h @ B  (matmul, same as original)
      - Init:     Uniform[-0.05, 0.05]  (original uses nn.init.uniform_ with ±0.05)
    """

    def __init__(self, hidden_dim: int, probe_rank: int = 256):
        super().__init__()
        # Matches original: nn.Parameter(torch.zeros(model_dim, probe_rank))
        self.proj = nn.Parameter(torch.zeros(hidden_dim, probe_rank))
        nn.init.uniform_(self.proj, -0.05, 0.05)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """Compute predicted pairwise squared distances.

        Args:
            h: (n_words, hidden_dim)
        Returns:
            dists: (n_words, n_words) predicted squared distances
        """
        # z = h @ B  →  (n_words, probe_rank)   [same as original: torch.matmul(batch, self.proj)]
        z = torch.matmul(h, self.proj)
        # All-pairs squared L2: ‖z[i] − z[j]‖²
        diff = z.unsqueeze(1) - z.unsqueeze(0)  # (n_words, n_words, probe_rank)
        return (diff ** 2).sum(-1)               # (n_words, n_words)


def probe_loss(pred_dists: torch.Tensor, gold_dists: torch.Tensor) -> torch.Tensor:
    """L1 loss between predicted and gold tree distances, normalised by |S|²."""
    n = pred_dists.shape[0]
    return torch.abs(pred_dists - gold_dists).sum() / (n ** 2)

## 6. UUAS Evaluation

In [ ]:
PUNCT_UPOS = {"PUNCT"}  # paper excludes punctuation from UUAS


def predicted_edges_from_distances(dist_matrix: np.ndarray) -> set:
    """Build MST from distance matrix and return its undirected edge set.

    scipy.sparse.csgraph.minimum_spanning_tree treats zeros as absent edges,
    which is correct here: diagonal (self-distances) are 0 and should be ignored.
    """
    sparse = csr_matrix(dist_matrix)
    mst = minimum_spanning_tree(sparse)
    mst_coo = mst.tocoo()
    # mst_coo.row / .col give the directed edges of the lower-triangular MST
    edges = {frozenset([int(i), int(j)]) for i, j in zip(mst_coo.row, mst_coo.col)}
    return edges


@torch.no_grad()
def compute_uuas(probe, data_with_hs, layer_idx, device):
    """Evaluate UUAS for a probe on a given layer.

    Protocol from the paper: punctuation (UPOS=PUNCT) and the ROOT relation
    are excluded from evaluation. MST is built on the non-punctuation sub-matrix.
    """
    probe.eval()
    total_correct = 0
    total_gold = 0

    for words, heads, dists, gold_edge_set, upos, word_states in data_with_hs:
        h = word_states[layer_idx].to(device)
        pred_dists = probe(h).cpu().numpy()    # (n_words, n_words)

        # Non-punctuation word indices (0-indexed)
        valid = [i for i, pos in enumerate(upos) if pos not in PUNCT_UPOS]
        if len(valid) < 2:
            continue

        # Gold edges restricted to valid (non-punct) tokens, excluding ROOT
        valid_set = set(valid)
        filtered_gold = {e for e in gold_edge_set if all(v in valid_set for v in e)}

        # Build MST on the valid sub-matrix
        sub_dist = pred_dists[np.ix_(valid, valid)]
        pred_sub_edges = predicted_edges_from_distances(sub_dist)

        # Remap sub-matrix indices back to original word indices
        # valid[sub_i] gives the original index for sub-matrix row/col sub_i
        pred_edges = {
            frozenset([valid[i], valid[j]])
            for edge in pred_sub_edges
            for i, j in [sorted(edge)]   # sorted() gives a stable (i, j) pair from the frozenset
        }

        total_correct += len(pred_edges & filtered_gold)
        total_gold    += len(filtered_gold)

    return total_correct / total_gold if total_gold > 0 else 0.0

## 7. Training

Hyperparameters from Appendix B.1 of the paper:
- Batch size: 32
- Max epochs: 300
- Early stopping: validation loss, patience 50
- LR: 1e-4, AdamW, linear decay with 10% warmup

In [ ]:
def get_linear_warmup_schedule(optimizer, warmup_steps, total_steps):
    """Linear warmup then linear decay to 0."""
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    return LambdaLR(optimizer, lr_lambda)


def train_probe_for_layer(
    layer_idx,
    train_data_hs,
    dev_data_hs,
    hidden_dim,
    probe_rank=256,
    lr=1e-4,
    max_epochs=300,
    patience=50,
    batch_size=32,
    device=DEVICE,
    verbose=False,
):
    probe = StructuralProbe(hidden_dim, probe_rank).to(device)
    optimizer = AdamW(probe.parameters(), lr=lr)

    steps_per_epoch = math.ceil(len(train_data_hs) / batch_size)
    total_steps = max_epochs * steps_per_epoch
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_warmup_schedule(optimizer, warmup_steps, total_steps)

    best_val_loss = float("inf")
    best_state = None
    no_improve = 0
    global_step = 0

    for epoch in range(max_epochs):
        probe.train()
        indices = torch.randperm(len(train_data_hs)).tolist()
        epoch_loss = 0.0

        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start: start + batch_size]
            optimizer.zero_grad()
            batch_loss = torch.tensor(0.0, device=device)

            for i in batch_idx:
                words, heads, dists, edges, upos, word_states = train_data_hs[i]
                h = word_states[layer_idx].to(device)
                gold = torch.tensor(dists, dtype=torch.float32, device=device)
                pred = probe(h)
                batch_loss = batch_loss + probe_loss(pred, gold)

            batch_loss = batch_loss / len(batch_idx)
            batch_loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_loss += batch_loss.item()
            global_step += 1

        # Validation loss
        probe.eval()
        val_loss = 0.0
        with torch.no_grad():
            for words, heads, dists, edges, upos, word_states in dev_data_hs:
                h = word_states[layer_idx].to(device)
                gold = torch.tensor(dists, dtype=torch.float32, device=device)
                pred = probe(h)
                val_loss += probe_loss(pred, gold).item()
        val_loss /= len(dev_data_hs)

        if verbose and epoch % 10 == 0:
            print(f"  Epoch {epoch:3d} | train={epoch_loss/steps_per_epoch:.4f} | val={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in probe.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"  Early stop at epoch {epoch}.")
                break

    probe.load_state_dict(best_state)
    return probe, best_val_loss

## 8. Layer Sweep

Train one probe per layer, pick the best by dev loss, report test UUAS.

The paper finds attachment score typically peaks in early-to-mid layers.

In [ ]:
# Set to a subset of layers for a quick test, or range(1, N_LAYERS+1) for all
LAYERS_TO_PROBE = list(range(1, N_LAYERS + 1))  # skip layer 0 (embedding)

layer_results = {}  # layer_idx -> {probe, dev_loss, dev_uuas}

for layer_idx in tqdm(LAYERS_TO_PROBE, desc="Layer sweep"):
    probe, dev_loss = train_probe_for_layer(
        layer_idx=layer_idx,
        train_data_hs=train_hs,
        dev_data_hs=dev_hs,
        hidden_dim=HIDDEN_DIM,
        probe_rank=256,
        lr=1e-4,
        max_epochs=300,
        patience=50,
        batch_size=32,
        device=DEVICE,
        verbose=False,
    )
    dev_uuas = compute_uuas(probe, dev_hs, layer_idx, DEVICE)
    layer_results[layer_idx] = {"probe": probe, "dev_loss": dev_loss, "dev_uuas": dev_uuas}
    tqdm.write(f"Layer {layer_idx:2d} | dev_loss={dev_loss:.4f} | dev_UUAS={dev_uuas:.4f}")

In [ ]:
# Select best layer by dev UUAS (paper selects by PTB test metric; we use dev)
best_layer = max(layer_results, key=lambda l: layer_results[l]["dev_uuas"])
best_probe = layer_results[best_layer]["probe"]

test_uuas = compute_uuas(best_probe, test_hs, best_layer, DEVICE)

print(f"\nBest layer: {best_layer}")
print(f"Dev  UUAS: {layer_results[best_layer]['dev_uuas']:.4f}")
print(f"Test UUAS: {test_uuas:.4f}")

## 9. Visualize UUAS per Layer

In [ ]:
layers = sorted(layer_results.keys())
dev_uuas_per_layer = [layer_results[l]["dev_uuas"] for l in layers]

plt.figure(figsize=(8, 4))
plt.plot(layers, dev_uuas_per_layer, marker="o", linewidth=2, markersize=6)
plt.axvline(best_layer, color="red", linestyle="--", label=f"Best layer ({best_layer})")
plt.xlabel("Layer")
plt.ylabel("UUAS")
plt.title(f"Structural Probe Dev UUAS per Layer — {MODEL_NAME}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{MODEL_NAME.replace('/', '_')}_uuas_per_layer.png", dpi=150)
plt.show()
print(f"Peak UUAS {max(dev_uuas_per_layer):.4f} at layer {best_layer}")

## 10. Inspect a Single Sentence

Visualize predicted vs. gold tree distances for a sample sentence.

In [ ]:
# Pick first test sentence with at least 6 words
sample = next(s for s in test_hs if len(s[0]) >= 6)
words, heads, gold_dists, gold_edges_set, upos, word_states = sample

best_probe.eval()
with torch.no_grad():
    h = word_states[best_layer].to(DEVICE)
    pred_dists = best_probe(h).cpu().numpy()

gold_d = np.array(gold_dists)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, mat, title in zip(axes, [gold_d, pred_dists], ["Gold distances", "Predicted distances"]):
    im = ax.imshow(mat, cmap="viridis")
    ax.set_xticks(range(len(words)))
    ax.set_yticks(range(len(words)))
    ax.set_xticklabels(words, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(words, fontsize=9)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.suptitle(" ".join(words), fontsize=11)
plt.tight_layout()
plt.show()